# 01 — Train the full dual-graph model (3 seeds)
**Not required to reproduce the reported numbers** — use `02_evaluate_main_and_twohop` with the released checkpoints for that. Training on a GPU is not bit-reproducible, so re-training gives close but not identical checkpoints. Output goes to a NEW directory; existing files are never overwritten.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = Path(BASE) / 'revision_checkpoints_2026-10-18'

import time
from sasp_gat.model import DualGraphSASPModel
from sasp_gat.training import train_with_early_stopping_av, checkpoint_path, save_checkpoint, vkey
assert torch.cuda.is_available(), 'GPU required'
OUT = Path(BASE) / 'repro_checkpoints'
OUT.mkdir(exist_ok=True)
# (use_spatial, use_ppi, name). Ablations: add (False, False, 'expression-only'), (True, False, 'expression+spatial'), (False, True, 'expression+PPI')
VARIANTS = [(True, True, 'full')]
rows = []
for use_sp, use_p, vn in VARIANTS:
    for seed in SEEDS:
        p = checkpoint_path(OUT, vkey(vn), seed)                  # refuses to overwrite
        torch.manual_seed(seed)                                   # seed first, then model construction
        model = DualGraphSASPModel(use_spatial=use_sp, use_ppi=use_p).to(device)
        t0 = time.time()
        best_state, best_val, best_epoch, hist = train_with_early_stopping_av(
            model, train_s, val_s, AVAIL, ppi_ei, device, log_every=25)
        save_checkpoint(p, best_state, vn, use_sp, use_p, seed, best_val, best_epoch, len(hist), LOGGED)
        rows.append(dict(variant=vn, seed=seed, best_val=best_val, best_epoch=best_epoch, epochs=len(hist), sec=time.time() - t0))
        print(rows[-1])
pd.DataFrame(rows).to_csv(OUT / 'training_results.csv', index=False)